# 06 — Stratégie & Dashboard (Person 6)

Brouillon d'exploration pour M7 (stratégie) et M8 (dashboard).
Croise `segment_profiles.csv`, `campaign_kpis.csv` et `churn_clv_predictions.csv`
pour produire des recommandations par segment.

Le code final et propre du dashboard est dans `dashboard/app.py`.
Ce notebook sert de brouillon d'exploration et documente le raisonnement.

In [1]:
import pandas as pd

customers = pd.read_csv("../data/processed/customers_clean.csv")
segments = pd.read_csv("../data/processed/customer_segments.csv")
profiles = pd.read_csv("../data/processed/segment_profiles.csv")
campaigns = pd.read_csv("../data/processed/campaign_kpis.csv")
churn = pd.read_csv("../data/processed/churn_clv_predictions.csv")

profiles

,Cluster,Persona_Name,Size_pct,Avg_Age,Dominant_Gender,Avg_Spend,Avg_Basket,Avg_Purchase_Frequency,Favorite_Category,Preferred_Channel,Persona_Description,Recommendation
0,0,Jeune Depensier Jules,50.0,32.5,Female,675.0,82.5,1.0,Accessories,In-Store,"Ce segment represente 50.0% de la clientele, a...",Cibler ce segment avec des offres et evenement...
1,1,Etudiante Econome Emma,25.0,22.0,Male,300.0,120.0,1.0,Outerwear,In-Store,"Ce segment represente 25.0% de la clientele, a...",Cibler ce segment avec des offres et evenement...
2,2,Etudiante Econome Emma,25.0,28.0,Female,500.0,95.0,2.0,Clothing,Online,"Ce segment represente 25.0% de la clientele, a...",Cibler ce segment avec des campagnes email et ...


## 1. Performance des campagnes par canal

In [2]:
roi_by_channel = campaigns.groupby("Channel")["ROI"].mean().sort_values(ascending=False)
roi_by_channel

Channel
Online      650.00
Email       400.00
Social      400.00
TV          316.67
In-Store    233.33
Name: ROI, dtype: float64

## 2. Churn moyen par segment (Cluster)

Attention : certains clients peuvent ne pas être classés (`Cluster` = `Unknown`)
s'ils sont absents de `customer_segments.csv` — à vérifier et signaler à Person 2.

In [3]:
unclassified = churn[~churn["Customer_ID"].isin(segments["Customer_ID"])]
print(f"Clients non classés dans customer_segments.csv : {len(unclassified)}")
unclassified

Clients non classés dans customer_segments.csv : 1


,Customer_ID,Cluster,Churn_Probability,Churn_Prediction,Risk_Segment
4,2005,Unknown,0.8683,1,High


In [4]:
churn_classified = churn[churn["Customer_ID"].isin(segments["Customer_ID"])].copy()
churn_classified["Cluster"] = churn_classified["Cluster"].astype(float).astype(int)
churn_by_cluster = churn_classified.groupby("Cluster")["Churn_Probability"].mean()
churn_by_cluster

Cluster
0    0.1662
1    0.1331
2    0.0706
Name: Churn_Probability, dtype: float64

## 3. Tableau croisé : segment × canal préféré × ROI × churn

In [5]:
strategy_table = profiles.copy()
strategy_table["ROI_Canal_Prefere"] = strategy_table["Preferred_Channel"].map(roi_by_channel)
strategy_table["Churn_Moyen"] = strategy_table["Cluster"].map(churn_by_cluster)

strategy_table[[
    "Cluster", "Persona_Name", "Size_pct", "Avg_Spend",
    "Preferred_Channel", "ROI_Canal_Prefere", "Churn_Moyen",
]]

,Cluster,Persona_Name,Size_pct,Avg_Spend,Preferred_Channel,ROI_Canal_Prefere,Churn_Moyen
0,0,Jeune Depensier Jules,50.0,675.0,In-Store,233.33,0.1662
1,1,Etudiante Econome Emma,25.0,300.0,In-Store,233.33,0.1331
2,2,Etudiante Econome Emma,25.0,500.0,Online,650.00,0.0706


## 4. Règle de priorisation (brouillon)

- Churn moyen > 15% ET taille importante → priorité rétention (🔴)
- ROI du canal préféré élevé (> 300%) ET churn faible → priorité investissement (🟢)
- Sinon → surveillance (🟡)

Client(s) non classé(s) avec churn élevé → anomalie à traiter en urgence, indépendamment
du raisonnement par segment (voir section 2).

In [6]:
def recommend(row):
    if row["Churn_Moyen"] is not None and row["Churn_Moyen"] > 0.15:
        return "🔴 Priorité rétention"
    if row["ROI_Canal_Prefere"] is not None and row["ROI_Canal_Prefere"] > 300:
        return "🟢 Priorité investissement"
    return "🟡 Surveillance"

strategy_table["Priorite_M7"] = strategy_table.apply(recommend, axis=1)
strategy_table[["Persona_Name", "Priorite_M7"]]

,Persona_Name,Priorite_M7
0,Jeune Depensier Jules,🔴 Priorité rétention
1,Etudiante Econome Emma,🟡 Surveillance
2,Etudiante Econome Emma,🟢 Priorité investissement


## 5. Suite

- Rédaction complète des recommandations → `reports/M7_digital_strategy.pdf`
- Version interactive de ces mêmes analyses → `dashboard/app.py`
- Fusion avec le travail de toute l'équipe → `reports/final_report.docx` (M9)